# 🎵 Spotify clustering

In this challenge, we'll be using a dataset from Spotify that contains metadata for songs on the platform.

By metadata we mean info about the song such as name, artists, metrics about it's sound and other musical attributes.

We will use this dataset to try and cluster songs together that are closely related!

This is the underlying theory behind how recommender algorithms work on sites such as Spotify, Netflix, etc.

## 🎯 What is Clustering?

**Clustering** is an *unsupervised learning* technique that groups similar data points together.

**Key differences from classification:**
- **No labels needed** - the algorithm finds patterns on its own
- **Discovers hidden structure** - useful when you don't know what you're looking for
- **Distance-based** - groups items that are "close" in feature space

**Real-world applications:**
- 🎵 Music recommendations (Spotify, Apple Music)
- 📺 Content suggestions (Netflix, YouTube)
- 🛍️ Customer segmentation (e-commerce)
- 📰 News article grouping

**Today's challenge:** Cluster songs by audio features to create auto-generated playlists!

Let's dive in! 👇

## Data Exploration

Please run the cell below to return the spotify song data!

In [ ]:
import pandas as pd

spotify_df = pd.read_csv('https://wagon-public-datasets.s3.amazonaws.com/Machine%20Learning%20Datasets/ML_spotify_data.csv')
spotify_df.head()

For the purposes of our analyses, we will only need the numeric features from our dataset. Select only these and save them in a variable called `spotify_numeric`

In [ ]:
# YOUR CODE HERE

Have a read through your features and try to understand what they are related to!

Spotify generate their own features that relate to abstract characteristics that can be attributed to a piece of music (e.g. 'valence' or 'danceability'), you don't need to worry about how these are calculated!

Then we also have some information that is more literal such as the 'key', 'tempo' and whether a song is 'explicit' or not.

Investigate the distributions of some of your variables below:

- What is the ratio of explicit vs non-explicit songs?
- How is popularity distributed?
- How are Spotify's internal song metrics distributed?

In [ ]:
import matplotlib.pyplot as plt

spotify_numeric.explicit.value_counts(normalize=True).plot(kind='bar')
plt.title('Explicit vs non-explicit');

In [ ]:
import seaborn as sns

sns.displot(spotify_numeric['popularity']);

In [ ]:
import numpy as np

fig, axs = plt.subplots(3, 2)

var_list = ['danceability', 'valence', 'energy', 'liveness', 'loudness', 'speechiness']

# Loop directly through the axs object and assign titles from the list
for row_axes, row_titles in zip(axs, np.array(var_list).reshape(3, 2)):
    for ax, title in zip(row_axes, row_titles):
        sns.histplot(x=spotify_numeric[title], ax=ax)

# Adjust layout to prevent overlap
plt.tight_layout()

# Show the plots
plt.show()

The cell below will visualize three of your features in 3D space. Feel free to switch up the variables that are being used for the *x*, *y*, *z* axes.

Because we are using plotly express, you can use your cursor to move around / zoom in & out of the chart.

In [ ]:
import plotly.express as px

fig = px.scatter_3d(spotify_numeric,
                    x='danceability',
                    y='energy',
                    z='speechiness',
                    opacity=0.7,
                    width=500,
                    height=500
           )
fig.show()

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'data_exploration',
    numeric_columns_count=len(spotify_numeric.columns),
    row_count=len(spotify_numeric),
    all_numeric=all(pd.api.types.is_numeric_dtype(spotify_numeric[col]) for col in spotify_numeric.columns)
)
result.write()
print(result.check())

## First model

Our goal in this challenge is to cluster our songs into similar groups! The plot above may or may not reveal things that look like clusters, but remember! We can only visualise three of our variables here at a time.

When we train a clustering model it will cluster our songs in n-dimensional space, where n is the number of features being fed into the model.

Let's start by instantiating a simple KMeans model, with 8 clusters  (this is an arbitrary starting point, we'll optimize this later!).

Fit this to your numeric spotify data and save the labels that your model has stored in a variable called `kmeans_8_labels`.

<details>
    <summary><i>Hint</i></summary>

To get the labels, have a look at the attributes your model has once it has been fitted to your data.
</details>

In [ ]:
# YOUR CODE HERE

What is the distribution of our labels? How many songs do we have in each cluster?

In [ ]:
# YOUR CODE HERE

We can also now visualise our songs in 3D space again, but this time colour them by our new labels to see what clusters we have created! Run the cell below to see how it's looking.

In [ ]:
fig = px.scatter_3d(spotify_numeric,
                    x='danceability',
                    y='energy',
                    z='speechiness',
                    color=kmeans_8_labels,
                    width=500,
                    height=500)
fig.show()

It looks a little bit chaotic doesn't it... I'm not sure I'd be forking out the monthly suscription costs if my discover weekly was as all over the place as this chart is.

Do you have any intuitions as to why our labels might look so poorly clustered?

<details>
    <summary><i>Answer</i></summary>

Remember that KMeans (and most unsupervised learning algorithms) are distance based. We have **not** scaled our numeric features yet. Perhaps doing this will make things look a bit clearer?
</details>

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'first_model',
    n_clusters=kmeans_simple.n_clusters,
    n_labels=len(kmeans_8_labels),
    n_songs=len(spotify_numeric),
    unique_labels=len(np.unique(kmeans_8_labels))
)
result.write()
print(result.check())

## Preprocessing

In [ ]:
# YOUR CODE HERE

## Modelling with preprocessed data

Now, let's train and fit a model in the same way that we did above. However, this time we will use the scaled data! Save the labels in a variable called `kmeans_8_scaled_labels`

In [ ]:
# YOUR CODE HERE

Run the cell below to see how our clusters look in 3D space, but with our newly scaled data.

In [ ]:
fig_scaled = px.scatter_3d(spotify_scaled,
                           x='danceability',
                           y='energy',
                           z='speechiness',
                           color=kmeans_8_scaled_labels,
                           width=500,
                           height=500)
fig_scaled.show()

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'preprocessing',
    original_shape=spotify_numeric.shape,
    scaled_shape=spotify_scaled.shape,
    scaled_n_clusters=kmeans_scaled.n_clusters,
    scaled_n_labels=len(kmeans_8_scaled_labels),
    n_songs=len(spotify_numeric)
)
result.write()
print(result.check())

## Finding the right value for *K*

It should look a bit more tidy, maybe a bit more stratified! Progress!

**However, it still doesn't look perfect**. Remember though, we are only looking at 3 dimensions out of the 10 dimensions that our model is trained on.

It might be that, if we could visualise 10 dimensionsal space, we would see some much more intuitively shaped clusters!

So far we have been using 8 clusters for our models so far, but we haven't tested whether this makes sense.

Let's use *the elbow method* to check how many of clusters we should ideally be using for this dataset. Do this below. Remember to use a plot to visualise your results.



In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

**📊 How to read the elbow plot:**

The "elbow" is where the curve bends sharply - adding more clusters beyond this point gives diminishing returns.

**What we're looking for:**
- Sharp drop initially = each cluster adds value
- Flattening curve = clusters stop improving model significantly
- The "elbow point" = optimal K

**In this plot:** The bend occurs around **K=5-6**. The exact value is subjective - both work! We'll use **5 clusters** for cleaner playlists.

👉 More clusters ≠ better! We want distinct groups, not over-segmentation.

## Creating a model with the ideal number of clusters

In [ ]:
spotify_clusters = 5

kmeans = KMeans(n_clusters=spotify_clusters, n_init='auto', max_iter=300)

kmeans.fit(spotify_scaled)

kmeans_optimal_labels = kmeans.labels_

fig_scaled = px.scatter_3d(spotify_scaled,
                           x='danceability',
                           y='energy',
                           z='speechiness',
                           color=kmeans_optimal_labels,
                           width=500,
                           height=500)
fig_scaled.show()

The chart doesn't reveal a whole lot more, but perhaps we can create some theoretical playlists based on our clusters?

Add the new labels from our model that has 5 clusters to our original spotify dataframe as a column called 'label'.

In [ ]:
# YOUR CODE HERE

## Generating Spotify playlists based on our clusters!

We should now see the original meta-data for our spotify songs, but **with the added label of which cluster they are located in** based on our KMeans algorithm

Let's generate 5 playlists (one for each cluster) that contains 15 random songs from that cluster.

Below we have created a dictionary called `daily_mixes`. Inside this dictionary we want to store keys that are the name of the cluster labels, and then as values we want dataframes that only contains the songs from that specific cluster.

Finish the for loop below to obtain this dictionary!

In [ ]:
daily_mixes = {}

for num_cluster in np.unique(kmeans_optimal_labels):
    pass  # YOUR CODE HERE

Run the cell below to print out our playlists!!!

In [ ]:
# we are using key + 1 below because the playlists are zero-indexed

for key, value in daily_mixes.items():
  print("-" * 50)
  print(f"Here are some songs for playlist {key + 1}")
  print("-" * 50)
  display(value.sample(5)[['name', 'artists']])

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'optimal_clusters',
    inertias=inertias,
    final_n_clusters=kmeans.n_clusters,
    n_playlists=len(daily_mixes),
    label_column_exists='label' in spotify_df.columns
)
result.write()
print(result.check())

## 🏁 What You Learned

**Core clustering workflow:**

1. Select numeric features
1. **Scale your data** (critical for distance-based algorithms!)
1. Use elbow method to find optimal K
1. Train KMeans and assign cluster labels
1. Apply clusters to real-world use case (playlists!)

**🏆 Key takeaways:**  
Clustering finds hidden patterns in data without labels. It's the foundation of recommendation systems, customer segmentation, and anomaly detection.

**⚠️ Important:** Results depend heavily on:

- Feature selection (what you measure)
- Scaling (necessary for distance-based methods)
- K value (too few = oversimplified, too many = noisy)

**Well done!** 🎉

### 🎁 Bonus: DBSCAN (Optional)

As a bonus, let's try and run a clustering analysis using [DBSCAN](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html)!

Remember, with `DBSCAN` we don't need to *a-prior* select the number of clusters we will end up with.'

Instantiate and fit a `DBSCAN` model. Read the documentation and be sure to trial out different values for `epsilon` and `min_samples`.

DBSCAN's performance depends heavily on `epsilon` (ε) - the maximum distance between two points to be considered neighbors. [This article](https://medium.com/@tarammullin/dbscan-parameter-estimation-ff8330e3a3bd) has some helpful tips on how to help pick reasonable values

We'll use a **k-distance graph** to estimate a good epsilon value. This is an advanced technique - don't worry if it seems complex!

**The idea:** Plot distances to k-nearest neighbors. The "elbow" in the curve suggests a good epsilon.

In [ ]:
# Finding the ideal value for epsilon

from sklearn.neighbors import NearestNeighbors

pass  # YOUR CODE HERE

In [ ]:
from sklearn.cluster import DBSCAN

pass  # YOUR CODE HERE

How many clusters has the model created? What is their distribution? Save your labels in a variable called `dbscan_labels`. Is this the same as what we came up with using the Elbow Method?

In [ ]:
# YOUR CODE HERE

**⚠️ Understanding DBSCAN labels:**

DBSCAN assigns **-1** to songs it considers **outliers** (noise). These songs don't fit well into any cluster.

- **Cluster 0, 1, 2...** = Meaningful groups
- **Cluster -1** = Outliers (too far from any cluster core)

This is different from KMeans, which forces every song into a cluster!

**Check your distribution:**
- How many songs are outliers (-1)?
- Is this reasonable for music data? (Unusual songs might naturally be outliers)

Run the cell below to plot your clusters using the DBSCAN labels.

In [ ]:
fig_dbscan = px.scatter_3d(spotify_scaled,
                           x='danceability',
                           y='energy',
                           z='speechiness',
                           color=dbscan_labels,
                           width=500,
                           height=500)
fig_dbscan.show()

Using your fitted model, add in your predicted cluster labels for each song to the spotify dataframe in a new column called 'label_dbscan'

<details>
    <summary><i>Hint</i></summary>

Your number of clusters will be very dependent on the parameters you specified when instantiating your model!
</details>

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

In [ ]:
# Let's see how many playlists we would get with our higher epsilon options

from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=1, cols=3, specs=[[{"type": "scene"}, {"type": "scene"}, {"type": "scene"}]], column_titles = epsilon)

for i, x in enumerate(epsilon):
  spotify_dbscan_higher_epsilons = DBSCAN(eps=x, min_samples=min_samples)

  spotify_dbscan_higher_epsilons.fit(spotify_scaled)

  dbscan_labels_higher_epsilons = spotify_dbscan_higher_epsilons.labels_

  np.unique(dbscan_labels_higher_epsilons, return_counts=True)

  fig.add_trace(go.Scatter3d(
                           x=spotify_scaled.danceability,
                           y=spotify_scaled.energy,
                           z=spotify_scaled.speechiness,
                           mode = 'markers',
                           marker = dict(color=dbscan_labels_higher_epsilons)),
                           row=1, col=i+1)

  print(pd.Series(dbscan_labels_higher_epsilons).value_counts())

fig.update_layout(showlegend=False)
fig.show()

The cell below will generate some new playlists using the DBSCAN clusters!

In [ ]:
daily_mixes_dbscan = {}

for num_cluster in np.unique(dbscan_labels):
    daily_mixes_dbscan[num_cluster] = spotify_df[spotify_df['label_dbscan'] == num_cluster]

for key, value in daily_mixes_dbscan.items():
    print("-" * 50)
    if key == -1:
        print(f"🔊 Outlier songs (DBSCAN noise cluster): {len(value)} songs")
    else:
        print(f"🎵 DBSCAN Cluster {key}: {len(value)} songs")
    print("-" * 50)

    # Only show sample if cluster has enough songs
    if len(value) >= 5:
        display(value.sample(5)[['name', 'artists']])
    else:
        display(value[['name', 'artists']])

### 🔍 KMeans vs DBSCAN: Which is Better?

**KMeans strengths:**
- ✅ Simple to understand and use
- ✅ Fast computation
- ✅ Every song gets a cluster
- ❌ Must specify K in advance
- ❌ Assumes spherical clusters

**DBSCAN strengths:**
- ✅ Finds clusters of arbitrary shape
- ✅ Automatically determines number of clusters
- ✅ Identifies outliers (-1 labels)
- ❌ Sensitive to epsilon/min_samples parameters
- ❌ Struggles with varying density clusters

**For Spotify playlists:**
KMeans is usually better! Music features tend to have relatively uniform density, and we want **every song** in a playlist (no outliers).

DBSCAN shines when you have irregular cluster shapes or need to identify anomalies.

## 🏁 Conclusion

You've just completed your first unsupervised clustering! **Congrats**! This is a *very* commonplace methodology, especially in recommender systems.

By no means is the example we have gone through meant to be perfect (especially with a subjective topic such as music + limited features), and it can churn out some pretty chaotic results, but **the principles will very much hold true for all clustering tasks**.

Importantly, *never forget to scale your data if you are using a distance-based algorithm*!

Finally, here are some links to more information about Spotify data / the Spotify API (perhaps some project inspiration):

- [Audio Analysis theory with the Spotify Web API](https://www.youtube.com/watch?v=goUzHd7cTuA)
- Spotify API [docs](https://developer.spotify.com/documentation/web-api/)
- Spotify API Wrappers [Tekore](https://github.com/felix-hilden/tekore) and [Spotipy](https://github.com/plamere/spotipy)